In [2]:
import sys
import os

# Add the root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(project_root)

In [3]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy import stats
from src.utils import calculate_rsi, calculate_macd, add_technical_indicators
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [5]:
# Load dataset
df = pd.read_csv(r'..\..\data\bitcoin\btcusd_1-min_data.csv')

In [6]:
# Which date has the highest closing price?
highest_close_date = df.loc[df['Close'].idxmax(), 'Timestamp']
highest_close_date = pd.to_datetime(highest_close_date, unit='s').date()
print(f"The date with the highest closing price is: {highest_close_date}")

The date with the highest closing price is: 2025-10-06


In [7]:
# Convert 'Timestamp' to datetime and set as index
df['Timestamp'] = pd.to_datetime(df['Timestamp'], unit='s')
df.set_index('Timestamp', inplace=True)
df = df.sort_index()
df.tail() 

,Open,High,Low,Close,Volume
Timestamp,,,,,
2025-10-19 00:12:00,107188.0,107188.0,107177.0,107188.0,0.581301
2025-10-19 00:13:00,107174.0,107174.0,107126.0,107126.0,0.814652
2025-10-19 00:14:00,107065.0,107140.0,107043.0,107140.0,0.058436
2025-10-19 00:15:00,107131.0,107171.0,107125.0,107146.0,0.438288
2025-10-19 00:16:00,107146.0,107146.0,107120.0,107120.0,0.282572


In [8]:
# Add technical indicators
new_df = add_technical_indicators(df)
new_df.tail()

,Open,High,Low,Close,Volume,RSI_14,MACD,MACD_Signal,EMA_20,SMA_50,BB_H,BB_L,ADX_14,OBV
Timestamp,,,,,,,,,,,,,,
2025-10-19 00:12:00,107188.0,107188.0,107177.0,107188.0,0.581301,51.210742,4.283051,9.835326,107182.404781,107156.50,107267.703379,107123.996621,18.456530,2.031751e+06
2025-10-19 00:13:00,107174.0,107174.0,107126.0,107126.0,0.814652,44.089075,-0.625183,7.743224,107177.032897,107157.40,107271.034115,107116.065885,17.976838,2.031750e+06
2025-10-19 00:14:00,107065.0,107140.0,107043.0,107140.0,0.058436,45.917994,-3.346732,5.525233,107173.505955,107158.48,107272.399244,107111.200756,18.802282,2.031750e+06
2025-10-19 00:15:00,107131.0,107171.0,107125.0,107146.0,0.438288,46.722354,-4.962228,3.427741,107170.886340,107158.78,107273.023809,107107.176191,18.874811,2.031750e+06
2025-10-19 00:16:00,107146.0,107146.0,107120.0,107120.0,0.282572,43.689956,-8.245454,1.093102,107166.040022,107159.12,107274.111283,107097.888717,19.015814,2.031750e+06


In [57]:
# Resampling to daily, monthly, yearly, and querterly frequency
df_daily = df.resample('D').mean()
df_monthly = df.resample('ME').mean()
df_yearly = df.resample('YE-DEC').mean() # Resampling the year to December end
df_quarterly = df.resample('QE-DEC').mean() # Resampling to quarterly frequency with December as the end of the quarter

Average Price / Typical Price: It smooths the volatility and better represents the market behavior.

In [58]:
# Compute the Average price
df['Average_Price'] = (df['Open'] + df['High'] + df['Low'] + df['Close']) / 4

In [ ]:
# ================================
# Predict Daily Return (Up/Down)
# ================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# --- Load Data ---
df = pd.read_csv(r'..\..\data\bitcoin\btcusd_1-min_data.csv', parse_dates=["Timestamp"], index_col="Timestamp")
df = add_technical_indicators(df)

# --- Create Target Variable (1 = Up, 0 = Down) ---
df["Return"] = df["Close"].pct_change()
df["Target"] = np.where(df["Return"].shift(-1) > 0, 1, 0)
df.dropna(inplace=True)

expected_features = [
    "Open", "High", "Low", "Close", "Volume",
    "RSI_14", "MACD", "MACD_Signal", "EMA_20",
    "SMA_50", "BB_H", "BB_L", "ADX_14", "OBV"
]

# use only those that exist
features = [f for f in expected_features if f in df.columns]
print("Using features:", features)

X = df[features]
y = df["Target"]

# --- Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# --- Standardize Features ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Define Models ---
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=20, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=20, learning_rate=0.05, random_state=42, use_label_encoder=False, eval_metric='logloss'),
    "SVM": SVC(kernel='rbf', probability=True),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

# --- Train & Evaluate ---
results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"\n{name} Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred))

# --- Summary ---
print("\n=== Model Comparison ===")
for name, acc in results.items():
    print(f"{name:20s}: {acc:.4f}")


C:\Users\miraz\AppData\Local\Temp\ipykernel_7700\585204556.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(r'..\..\data\bitcoin\btcusd_1-min_data.csv', parse_dates=["Timestamp"], index_col="Timestamp")


Using features: ['Open', 'High', 'Low', 'Close', 'Volume', 'RSI_14', 'MACD', 'MACD_Signal', 'EMA_20', 'SMA_50', 'BB_H', 'BB_L', 'ADX_14', 'OBV']

Logistic Regression Accuracy: 0.5013
              precision    recall  f1-score   support

           0       0.55      0.40      0.46    785076
           1       0.47      0.62      0.53    666086

    accuracy                           0.50   1451162
   macro avg       0.51      0.51      0.50   1451162
weighted avg       0.51      0.50      0.50   1451162


Random Forest Accuracy: 0.5205
              precision    recall  f1-score   support

           0       0.54      0.75      0.63    785076
           1       0.46      0.25      0.32    666086

    accuracy                           0.52   1451162
   macro avg       0.50      0.50      0.48   1451162
weighted avg       0.50      0.52      0.49   1451162



c:\Users\miraz\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:46:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost Accuracy: 0.5410


c:\Users\miraz\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\miraz\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\miraz\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

              precision    recall  f1-score   support

           0       0.54      1.00      0.70    785076
           1       0.00      0.00      0.00    666086

    accuracy                           0.54   1451162
   macro avg       0.27      0.50      0.35   1451162
weighted avg       0.29      0.54      0.38   1451162

